# Transcription Factor Discovery and Ranking for M(KP) Polarization

## Notebook Overview

**Objective**: Identify and rank transcription factors (TFs) that drive M(KP) macrophage polarization through de novo computational discovery, extending beyond the known STAT6 pathway.

**Research Question**: What are the key transcriptional regulators and TF combinations that induce M(KP) cell state conversion, and can we predict minimal sufficient TF sets for therapeutic targeting?

## Scientific Rationale

### Background
Our previous analysis (Notebook 6) successfully validated M(KP) as a STAT6-dependent polarization state through trajectory analysis. However, this approach was **literature-guided validation** rather than **de novo discovery**. To advance therapeutic targeting and mechanistic understanding, we need to:

1. **Discover novel TF regulators** beyond known STAT6 pathway components
2. **Rank TF importance** for M(KP) conversion efficiency 
3. **Identify minimal TF combinations** sufficient for state conversion
4. **Predict therapeutic targets** through computational perturbation analysis

### Methodological Innovation
This analysis represents a shift from **hypothesis-driven validation** to **data-driven discovery**, using three complementary computational approaches:

- **Gene Regulatory Network Inference**: Discover TF-target relationships from expression data
- **Trajectory-Based TF Activity**: Quantify TF influence on cell fate transitions
- **Machine Learning Prediction**: Rank TFs by predictive power for state classification

## Analytical Framework

### Input Data
- **Source**: Processed AnnData from Notebook 6 (post-trajectory analysis)
- **Target Population**: Cluster 2 (M(KP) cells) vs other IM clusters
- **Resolution**: Leiden 1.1 (optimal M(KP) signal concentration)
- **Cell Numbers**: 165 M(KP) cells (84 KP+, 81 KP-) vs 1,667 other IM cells

### Expected Outcomes
1. **Novel TF candidates** for M(KP) induction beyond STAT6
2. **Ranked TF importance** scores across multiple analytical methods
3. **Minimal TF combinations** for efficient M(KP) conversion
4. **Therapeutic target prioritization** based on druggability and network centrality
5. **Validation** against known M(KP) biology and STAT6 pathway components

---

## Section 1: Gene Regulatory Network Inference with SCENIC

### Methodology Overview

**SCENIC (Single-Cell rEgulatory Network Inference and Clustering)** reconstructs gene regulatory networks from single-cell expression data through three steps:

1. **Co-expression Module Detection**: Identify genes co-expressed with each TF
2. **Regulon Pruning**: Retain only TF-target pairs with predicted binding sites
3. **Regulon Activity Scoring**: Calculate TF activity in each cell using AUCell

### Scientific Logic

**Why SCENIC for M(KP) Discovery?**
- **Unbiased TF identification**: No prior assumptions about which TFs are important
- **Mechanistic insights**: Links TFs to their predicted target genes
- **Cell-type specificity**: Identifies TFs active specifically in M(KP) cells
- **Therapeutic relevance**: TFs are druggable targets (small molecules, degraders)

**Key Innovation**: While our previous analysis focused on STAT6 pathway *targets*, SCENIC will identify the full *regulatory hierarchy* controlling M(KP) conversion.

### Analysis Plan

#### Step 1.1: GENIE3 Co-expression Network Construction
- Use Random Forest regression to predict each gene from all TFs
- Extract TF-target importance scores
- Filter for high-confidence regulatory relationships

#### Step 1.2: Motif-Based Regulon Pruning  
- Query TF binding site databases (JASPAR, ENCODE)
- Retain only TF-target pairs with predicted binding sites
- Create curated regulons for each TF

#### Step 1.3: AUCell Regulon Activity Scoring
- Calculate TF activity scores in each cell
- Compare regulon activity: M(KP) vs other IM clusters
- Identify TFs with significantly higher activity in M(KP) cells

#### Step 1.4: M(KP)-Specific TF Ranking
- Rank TFs by differential regulon activity (M(KP) vs others)
- Validate against known M(KP) markers (Cxcl16, Mmp14, Il1rn)
- Identify novel TF candidates for experimental validation

### Expected Results
- **Top 10-20 TFs** driving M(KP) conversion
- **Regulon compositions** for each M(KP)-associated TF
- **Target gene networks** explaining M(KP) molecular signature
- **Novel therapeutic targets** beyond STAT6 pathway

In [1]:
# Section 1: SCENIC Analysis Setup
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# SCENIC imports
# Note: May need installation: pip install pyscenic
try:
    import pyscenic as ps
    from pyscenic.utils import modules_from_adjacencies
    from pyscenic.prune import prune2df
    from pyscenic.aucell import aucell
    print("SCENIC modules imported successfully")
except ImportError:
    print("SCENIC not installed. Install with: pip install pyscenic")
    print("Proceeding with alternative GRN inference methods...")

# Load processed data from Notebook 6
print("Loading trajectory-analyzed data...")
# adata = sc.read_h5ad('trajectory_analysis_results.h5ad')  # Adjust path as needed

# Filter for Interstitial Macrophages (IM) only - our cell type of interest
# adata_im = adata[adata.obs['cell_type'] == 'IM'].copy()
# print(f"IM cells for TF analysis: {adata_im.n_obs}")

SCENIC not installed. Install with: pip install pyscenic
Proceeding with alternative GRN inference methods...
Loading trajectory-analyzed data...


In [ ]:
# Step 1.1: Prepare expression matrix for SCENIC

def prepare_scenic_input(adata, min_genes=200, min_cells=3):
    """
    Prepare expression matrix for SCENIC analysis
    
    Parameters:
    - min_genes: minimum genes per cell
    - min_cells: minimum cells per gene
    """
    print("Preparing SCENIC input matrix...")
    
    # Use raw counts for SCENIC (important!)
    if 'counts' in adata.layers:
        expression_matrix = adata.layers['counts']
    else:
        print("Warning: Using current X matrix - ensure these are raw counts")
        expression_matrix = adata.X
    
    # Convert to DataFrame for SCENIC
    expr_df = pd.DataFrame(
        expression_matrix.toarray() if hasattr(expression_matrix, 'toarray') else expression_matrix,
        index=adata.obs_names,
        columns=adata.var_names
    ).T  # SCENIC expects genes as rows, cells as columns
    
    print(f"Expression matrix shape: {expr_df.shape}")
    print(f"Genes: {expr_df.shape[0]}, Cells: {expr_df.shape[1]}")
    
    return expr_df

# Uncomment when data is loaded:
# expr_matrix = prepare_scenic_input(adata_im)

In [ ]:
# Step 1.2: GENIE3 Network Inference

def run_genie3_analysis(expr_matrix, tf_list=None, n_workers=4):
    """
    Run GENIE3 to infer TF-target relationships
    
    Parameters:
    - expr_matrix: genes x cells DataFrame
    - tf_list: list of TF genes (if None, will use default TF database)
    - n_workers: parallel processing cores
    """
    print("Running GENIE3 network inference...")
    
    # Load TF list (update path to actual TF database)
    if tf_list is None:
        # Example TF list - replace with actual mouse TF database
        # tf_database = pd.read_csv('mouse_tfs.txt', header=None)[0].tolist()
        # tf_list = [tf for tf in tf_database if tf in expr_matrix.index]
        print("Using all genes as potential TFs (will filter later)")
        tf_list = expr_matrix.index.tolist()
    
    print(f"TFs for analysis: {len(tf_list)}")
    
    # Run GENIE3
    try:
        adjacencies = ps.grnboost2(
            expr_matrix,
            tf_names=tf_list,
            verbose=True,
            seed=42
        )
        print(f"GENIE3 completed. Adjacencies shape: {adjacencies.shape}")
        return adjacencies
        
    except Exception as e:
        print(f"GENIE3 failed: {e}")
        print("Consider using alternative GRN inference methods")
        return None

# Uncomment when ready:
# adjacencies = run_genie3_analysis(expr_matrix)

---

## Section 2: Trajectory-Based Transcription Factor Activity Analysis

### Methodology Overview

**Approach**: Quantify TF activity changes along the cellular trajectories we identified in Notebook 6, focusing on transitions leading to M(KP) fate.

### Scientific Logic

**Why Trajectory-Based TF Analysis?**
- **Temporal dynamics**: Captures TF activity changes during M(KP) conversion process
- **Transition specificity**: Identifies TFs crucial for fate commitment vs maintenance  
- **Predictive power**: TFs with changing activity predict cell fate transitions
- **Mechanistic ordering**: Reveals early vs late TF regulators in conversion process

**Integration with Previous Work**: This extends our PAGA trajectory analysis by adding **regulatory mechanism discovery** to the **cell fate mapping** we already established.

### Analysis Plan

#### Step 2.1: TF Activity Score Calculation
- Calculate TF activity using multiple methods:
  - **Direct expression**: TF mRNA levels
  - **Target gene expression**: Average expression of known TF targets
  - **Regulon activity scores**: From SCENIC results (Section 1)

#### Step 2.2: Pseudotime TF Activity Profiling
- Map TF activity along pseudotime trajectories
- Identify TFs with:
  - **Early activation**: High activity in trajectory origins
  - **Late activation**: High activity in M(KP) terminal states
  - **Transition peaks**: Activity spikes during fate commitment

#### Step 2.3: Branch Point TF Analysis
- Focus on trajectory branch points leading to M(KP) vs other fates
- Identify TFs that:
  - **Drive M(KP) commitment**: Higher activity in M(KP)-bound trajectories
  - **Inhibit alternative fates**: Lower activity in non-M(KP) branches
  - **Act as fate switches**: Sharp activity changes at decision points

#### Step 2.4: TF Transition Probability Modeling
- Use CellRank or similar tools to model transition probabilities
- Rank TFs by their influence on M(KP) transition likelihood
- Identify TFs whose expression levels predict M(KP) fate commitment

### Expected Results
- **Temporal TF hierarchy**: Early, intermediate, and late M(KP) regulators
- **Transition-critical TFs**: Factors essential for M(KP) fate commitment
- **Dynamic TF profiles**: Activity patterns distinguishing M(KP) from other trajectories
- **Predictive TF signatures**: Minimal TF sets predicting M(KP) conversion

In [ ]:
# Section 2: Trajectory-Based TF Activity Analysis

def calculate_tf_activity_scores(adata, tf_target_dict=None, method='expression'):
    """
    Calculate TF activity scores using different methods
    
    Parameters:
    - adata: AnnData object with trajectory information
    - tf_target_dict: dictionary mapping TFs to target genes
    - method: 'expression', 'targets', or 'regulon'
    """
    print(f"Calculating TF activity scores using method: {method}")
    
    if method == 'expression':
        # Simple: use TF expression levels as activity proxy
        # Load mouse TF list (placeholder - replace with actual database)
        # mouse_tfs = pd.read_csv('mouse_tf_list.txt')['gene_symbol'].tolist()
        # tf_genes = [tf for tf in mouse_tfs if tf in adata.var_names]
        
        # For now, use genes containing common TF keywords
        tf_keywords = ['Stat', 'Fox', 'Nf', 'Jun', 'Fos', 'Irf', 'Rel', 'Klf', 'Ets', 'Hif']
        tf_genes = [gene for gene in adata.var_names 
                   if any(keyword.lower() in gene.lower() for keyword in tf_keywords)]
        
        print(f"Identified {len(tf_genes)} potential TF genes")
        
        # Extract TF expression matrix
        tf_activity = adata[:, tf_genes].X.toarray() if hasattr(adata.X, 'toarray') else adata[:, tf_genes].X
        tf_activity_df = pd.DataFrame(tf_activity, 
                                    index=adata.obs_names, 
                                    columns=tf_genes)
        
        return tf_activity_df
    
    elif method == 'targets':
        print("Target-based TF activity calculation not implemented yet")
        print("Will implement after SCENIC regulon discovery")
        return None
    
    elif method == 'regulon':
        print("Regulon-based TF activity requires SCENIC results")
        print("Will implement after Section 1 completion")
        return None

# Uncomment when data is loaded:
# tf_activity_df = calculate_tf_activity_scores(adata_im, method='expression')

In [ ]:
# Step 2.2: Pseudotime TF Activity Profiling

def analyze_tf_pseudotime_dynamics(adata, tf_activity_df, pseudotime_key='dpt_pseudotime'):
    """
    Analyze TF activity changes along pseudotime
    
    Parameters:
    - adata: AnnData with pseudotime information
    - tf_activity_df: TF activity scores DataFrame
    - pseudotime_key: column name for pseudotime values
    """
    print("Analyzing TF activity dynamics along pseudotime...")
    
    if pseudotime_key not in adata.obs.columns:
        print(f"Error: {pseudotime_key} not found in adata.obs")
        print("Available pseudotime keys:", [col for col in adata.obs.columns if 'time' in col.lower()])
        return None
    
    # Combine pseudotime with TF activities
    pseudotime = adata.obs[pseudotime_key].values
    
    # Calculate correlations between TF activity and pseudotime
    tf_pseudotime_corr = []
    
    for tf in tf_activity_df.columns:
        # Remove NaN values
        valid_mask = ~(np.isnan(pseudotime) | np.isnan(tf_activity_df[tf]))
        
        if valid_mask.sum() > 10:  # Minimum cells for correlation
            corr = np.corrcoef(pseudotime[valid_mask], tf_activity_df[tf][valid_mask])[0, 1]
            tf_pseudotime_corr.append({
                'TF': tf,
                'pseudotime_correlation': corr,
                'abs_correlation': abs(corr)
            })
    
    # Create results DataFrame
    tf_dynamics_df = pd.DataFrame(tf_pseudotime_corr)
    tf_dynamics_df = tf_dynamics_df.sort_values('abs_correlation', ascending=False)
    
    print(f"Analyzed {len(tf_dynamics_df)} TFs for pseudotime dynamics")
    print("\nTop 10 TFs by pseudotime correlation:")
    print(tf_dynamics_df.head(10))
    
    return tf_dynamics_df

# Uncomment when ready:
# tf_dynamics = analyze_tf_pseudotime_dynamics(adata_im, tf_activity_df)

---

## Section 3: Machine Learning Transcription Factor Prediction

### Methodology Overview

**Approach**: Train machine learning models to predict M(KP) cell state from TF expression profiles, then extract TF importance scores for ranking.

### Scientific Logic

**Why ML for TF Discovery?**
- **Predictive power**: Identifies TFs that best distinguish M(KP) from other states
- **Feature importance**: Quantifies each TF's contribution to accurate classification
- **Minimal gene sets**: Finds smallest TF combinations for state prediction
- **Therapeutic relevance**: High-importance TFs are priority intervention targets

**Model Selection Rationale**:
- **Random Forest**: Handles feature interactions, provides importance scores
- **Logistic Regression**: Linear interpretability, coefficient-based ranking
- **SVM**: Non-linear decision boundaries, kernel-based feature weighting
- **Neural Networks**: Complex interactions, gradient-based importance

### Analysis Plan

#### Step 3.1: Dataset Preparation
- **Positive class**: M(KP) cells (Cluster 2, n=165)
- **Negative class**: Other IM cells (Clusters 0,1,3,4,5, n=1,667)
- **Features**: TF expression values (log-normalized)
- **Train/test split**: 80/20 with stratification

#### Step 3.2: Multi-Model Training and Validation
- Train multiple ML models with cross-validation
- Optimize hyperparameters for each model type
- Evaluate performance: AUC, precision, recall, F1-score
- Select best-performing model for TF ranking

#### Step 3.3: Feature Importance Extraction
- **Random Forest**: Feature importance scores
- **Logistic Regression**: Coefficient magnitudes
- **SVM**: Support vector contributions
- **Neural Networks**: Permutation importance or SHAP values

#### Step 3.4: Minimal TF Set Discovery
- Use recursive feature elimination to find minimal TF combinations
- Test prediction accuracy with top 5, 10, 15, 20 TFs
- Identify point of diminishing returns for TF addition
- Validate minimal sets against biological knowledge

### Expected Results
- **Model performance metrics**: Classification accuracy for M(KP) prediction
- **TF importance rankings**: Quantitative scores for each TF's predictive value
- **Minimal TF combinations**: Smallest gene sets achieving high accuracy
- **Cross-method validation**: Consensus TF rankings across ML approaches

In [ ]:
# Section 3: Machine Learning TF Prediction

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, SelectKBest, f_classif

def prepare_ml_dataset(adata, tf_activity_df, mkp_cluster='2'):
    """
    Prepare dataset for ML-based TF importance analysis
    
    Parameters:
    - adata: AnnData object with cluster annotations
    - tf_activity_df: TF activity scores
    - mkp_cluster: cluster ID for M(KP) cells
    """
    print("Preparing ML dataset for TF importance analysis...")
    
    # Create binary labels: M(KP) vs others
    cluster_key = 'leiden_1.1'  # Adjust based on your cluster column name
    
    if cluster_key not in adata.obs.columns:
        print(f"Error: {cluster_key} not found. Available cluster columns:")
        print([col for col in adata.obs.columns if 'leiden' in col.lower()])
        return None, None
    
    # Create labels
    labels = (adata.obs[cluster_key] == mkp_cluster).astype(int)
    
    # Align TF activity data with labels
    common_cells = tf_activity_df.index.intersection(adata.obs_names)
    X = tf_activity_df.loc[common_cells].values
    y = labels.loc[common_cells].values
    
    print(f"Dataset prepared:")
    print(f"  Total cells: {len(y)}")
    print(f"  M(KP) cells (positive): {y.sum()}")
    print(f"  Other cells (negative): {len(y) - y.sum()}")
    print(f"  Features (TFs): {X.shape[1]}")
    print(f"  Class balance: {y.mean():.3f}")
    
    return X, y, tf_activity_df.columns.tolist()

# Uncomment when ready:
# X, y, tf_names = prepare_ml_dataset(adata_im, tf_activity_df)

In [ ]:
# Step 3.2: Multi-Model Training and Evaluation

def train_tf_prediction_models(X, y, tf_names, test_size=0.2, random_state=42):
    """
    Train multiple ML models to predict M(KP) state from TF expression
    
    Parameters:
    - X: TF expression features
    - y: M(KP) binary labels
    - tf_names: list of TF names
    - test_size: fraction for test set
    - random_state: reproducibility seed
    """
    print("Training multiple ML models for TF importance ranking...")
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Scale features for models that require it
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Initialize models
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=random_state, 
                                             class_weight='balanced'),
        'LogisticRegression': LogisticRegression(random_state=random_state, class_weight='balanced',
                                               max_iter=1000),
        'SVM': SVC(random_state=random_state, class_weight='balanced', probability=True),
        'MLP': MLPClassifier(random_state=random_state, max_iter=500)
    }
    
    # Train and evaluate models
    model_results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        
        # Use scaled data for models that need it
        if name in ['LogisticRegression', 'SVM', 'MLP']:
            X_tr, X_te = X_train_scaled, X_test_scaled
        else:
            X_tr, X_te = X_train, X_test
        
        # Train model
        model.fit(X_tr, y_train)
        
        # Predictions
        y_pred = model.predict(X_te)
        y_pred_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
        
        # Evaluate
        cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='roc_auc')
        test_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
        
        model_results[name] = {
            'model': model,
            'cv_auc_mean': cv_scores.mean(),
            'cv_auc_std': cv_scores.std(),
            'test_auc': test_auc,
            'scaler': scaler if name in ['LogisticRegression', 'SVM', 'MLP'] else None
        }
        
        print(f"  CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
        if test_auc:
            print(f"  Test AUC: {test_auc:.3f}")
    
    return model_results, X_train, X_test, y_train, y_test

# Uncomment when ready:
# model_results, X_train, X_test, y_train, y_test = train_tf_prediction_models(X, y, tf_names)

In [ ]:
# Step 3.3: Feature Importance Extraction

def extract_tf_importance_scores(model_results, tf_names):
    """
    Extract TF importance scores from trained models
    
    Parameters:
    - model_results: dictionary of trained models
    - tf_names: list of TF gene names
    """
    print("Extracting TF importance scores from models...")
    
    importance_results = {}
    
    for model_name, result in model_results.items():
        model = result['model']
        
        if model_name == 'RandomForest':
            # Feature importance from Random Forest
            importance = model.feature_importances_
            
        elif model_name == 'LogisticRegression':
            # Coefficient magnitudes from Logistic Regression
            importance = np.abs(model.coef_[0])
            
        elif model_name == 'SVM':
            # For linear SVM, use coefficient magnitudes
            if hasattr(model, 'coef_'):
                importance = np.abs(model.coef_[0])
            else:
                print(f"  {model_name}: Cannot extract importance (non-linear kernel)")
                continue
                
        elif model_name == 'MLP':
            # For MLP, use first layer weight magnitudes (simplified)
            if hasattr(model, 'coefs_'):
                importance = np.abs(model.coefs_[0]).sum(axis=1)
            else:
                print(f"  {model_name}: Cannot extract importance")
                continue
        
        # Create importance DataFrame
        importance_df = pd.DataFrame({
            'TF': tf_names,
            'importance': importance,
            'importance_normalized': importance / importance.max()
        }).sort_values('importance', ascending=False)
        
        importance_results[model_name] = importance_df
        
        print(f"\n{model_name} - Top 10 TFs:")
        print(importance_df.head(10)[['TF', 'importance_normalized']].to_string(index=False))
    
    return importance_results

# Uncomment when ready:
# tf_importance_results = extract_tf_importance_scores(model_results, tf_names)

---

## Section 4: Integrated Transcription Factor Ranking and Validation

### Methodology Overview

**Approach**: Integrate results from SCENIC, trajectory analysis, and machine learning to create a consensus ranking of TFs driving M(KP) conversion.

### Integration Strategy

#### Step 4.1: Cross-Method TF Ranking Consensus
- **Rank aggregation**: Combine TF rankings from all three methods
- **Weighted scoring**: Assign weights based on method reliability and biological relevance
- **Statistical validation**: Test ranking stability across methods
- **Consensus score calculation**: Develop integrated TF importance metric

#### Step 4.2: Biological Validation Framework
- **Known M(KP) markers**: Validate against Cxcl16, Mmp14, Il1rn target predictions
- **STAT6 pathway**: Confirm STAT6 appears in top-ranked TFs
- **Literature validation**: Cross-reference with published macrophage TF studies
- **Pathway coherence**: Ensure top TFs connect to known M(KP) biology

#### Step 4.3: Novel TF Discovery and Characterization
- **Literature gap analysis**: Identify high-ranking TFs not previously linked to M(KP)
- **Target gene analysis**: Characterize predicted targets of novel TFs
- **Pathway enrichment**: Test if novel TF targets enrich for M(KP)-relevant pathways
- **Druggability assessment**: Evaluate therapeutic targeting potential

#### Step 4.4: Minimal TF Combination Optimization
- **Combinatorial analysis**: Test all combinations of top-ranked TFs
- **Synergy detection**: Identify TF pairs/triplets with enhanced predictive power
- **Redundancy removal**: Eliminate TFs with overlapping regulatory functions
- **Therapeutic prioritization**: Rank combinations by intervention feasibility

### Validation Criteria

**Success Metrics**:
1. **STAT6 validation**: STAT6 appears in top 10 TFs across methods
2. **Known marker coherence**: Top TFs predict known M(KP) markers as targets
3. **Cross-method consistency**: >50% overlap in top 20 TFs between methods
4. **Literature support**: Top 5 novel TFs have macrophage/immune function evidence
5. **Minimal set performance**: <10 TFs achieve >90% M(KP) classification accuracy

### Expected Deliverables
- **Consensus TF ranking**: Top 50 TFs with integrated importance scores
- **Novel TF candidates**: 5-10 previously unknown M(KP) regulators for validation
- **Minimal TF combinations**: Optimal gene sets for M(KP) conversion/targeting
- **Therapeutic targets**: Priority TFs for drug development efforts
- **Mechanistic model**: Updated M(KP) regulatory network beyond STAT6 pathway

In [ ]:
# Section 4: Integrated TF Ranking and Validation

def integrate_tf_rankings(scenic_results=None, trajectory_results=None, ml_results=None, 
                         weights={'scenic': 0.4, 'trajectory': 0.3, 'ml': 0.3}):
    """
    Integrate TF rankings from multiple methods into consensus scores
    
    Parameters:
    - scenic_results: TF rankings from SCENIC analysis
    - trajectory_results: TF rankings from trajectory analysis  
    - ml_results: TF rankings from ML analysis
    - weights: relative weights for each method
    """
    print("Integrating TF rankings from multiple methods...")
    
    # Collect all TFs mentioned across methods
    all_tfs = set()
    
    if scenic_results is not None:
        all_tfs.update(scenic_results['TF'].tolist())
        print(f"SCENIC TFs: {len(scenic_results)}")
    
    if trajectory_results is not None:
        all_tfs.update(trajectory_results['TF'].tolist())
        print(f"Trajectory TFs: {len(trajectory_results)}")
    
    if ml_results is not None:
        # Handle multiple ML models
        for model_name, df in ml_results.items():
            all_tfs.update(df['TF'].tolist())
        print(f"ML TFs: {len(all_tfs)}")
    
    print(f"Total unique TFs across methods: {len(all_tfs)}")
    
    # Create integrated ranking DataFrame
    integrated_scores = []
    
    for tf in all_tfs:
        scores = {'TF': tf}
        
        # SCENIC score (normalized regulon activity difference)
        if scenic_results is not None and tf in scenic_results['TF'].values:
            scenic_rank = scenic_results[scenic_results['TF'] == tf].index[0] + 1
            scores['scenic_score'] = 1.0 / scenic_rank  # Higher rank = higher score
        else:
            scores['scenic_score'] = 0.0
        
        # Trajectory score (pseudotime correlation)
        if trajectory_results is not None and tf in trajectory_results['TF'].values:
            traj_score = trajectory_results[trajectory_results['TF'] == tf]['abs_correlation'].iloc[0]
            scores['trajectory_score'] = traj_score
        else:
            scores['trajectory_score'] = 0.0
        
        # ML score (average across models)
        if ml_results is not None:
            ml_scores = []
            for model_name, df in ml_results.items():
                if tf in df['TF'].values:
                    ml_score = df[df['TF'] == tf]['importance_normalized'].iloc[0]
                    ml_scores.append(ml_score)
            scores['ml_score'] = np.mean(ml_scores) if ml_scores else 0.0
        else:
            scores['ml_score'] = 0.0
        
        # Calculate integrated score
        integrated_score = (
            weights['scenic'] * scores['scenic_score'] +
            weights['trajectory'] * scores['trajectory_score'] +
            weights['ml'] * scores['ml_score']
        )
        
        scores['integrated_score'] = integrated_score
        scores['methods_detected'] = sum([scores['scenic_score'] > 0, 
                                        scores['trajectory_score'] > 0,
                                        scores['ml_score'] > 0])
        
        integrated_scores.append(scores)
    
    # Create final DataFrame
    consensus_df = pd.DataFrame(integrated_scores)
    consensus_df = consensus_df.sort_values('integrated_score', ascending=False)
    
    print(f"\nIntegrated TF ranking completed. Top 20 TFs:")
    print(consensus_df.head(20)[['TF', 'integrated_score', 'methods_detected']].to_string(index=False))
    
    return consensus_df

# Placeholder for integration (uncomment when all sections complete):
# consensus_tf_ranking = integrate_tf_rankings(
#     scenic_results=scenic_tf_ranking,
#     trajectory_results=tf_dynamics,
#     ml_results=tf_importance_results
# )

In [ ]:
# Step 4.2: Validation Against Known M(KP) Biology

def validate_tf_rankings(consensus_df, known_mkp_markers=['Cxcl16', 'Mmp14', 'Il1rn', 'Isg15', 'Ifi205']):
    """
    Validate TF rankings against known M(KP) biology
    
    Parameters:
    - consensus_df: integrated TF ranking results
    - known_mkp_markers: list of validated M(KP) marker genes
    """
    print("Validating TF rankings against known M(KP) biology...")
    
    validation_results = {}
    
    # Check if STAT6 is highly ranked
    stat6_tfs = [tf for tf in consensus_df['TF'] if 'Stat6' in tf]
    if stat6_tfs:
        stat6_rank = consensus_df[consensus_df['TF'].isin(stat6_tfs)].index[0] + 1
        validation_results['STAT6_rank'] = stat6_rank
        validation_results['STAT6_top10'] = stat6_rank <= 10
        print(f"STAT6 ranking: {stat6_rank} (Top 10: {validation_results['STAT6_top10']})")
    else:
        print("Warning: STAT6 not found in TF rankings")
        validation_results['STAT6_rank'] = None
        validation_results['STAT6_top10'] = False
    
    # Check for other known immune/macrophage TFs
    known_immune_tfs = ['Nfkb1', 'Rel', 'Rela', 'Jun', 'Fos', 'Irf1', 'Irf3', 'Irf7', 'Irf8',
                       'Stat1', 'Stat3', 'Hif1a', 'Klf2', 'Klf4', 'Foxo1', 'Foxo3']
    
    found_immune_tfs = []
    for tf in known_immune_tfs:
        matches = [t for t in consensus_df['TF'] if tf.lower() in t.lower()]
        if matches:
            rank = consensus_df[consensus_df['TF'] == matches[0]].index[0] + 1
            found_immune_tfs.append({'TF': matches[0], 'rank': rank})
    
    validation_results['known_immune_tfs'] = found_immune_tfs
    
    print(f"\nKnown immune TFs found in ranking:")
    for tf_info in found_immune_tfs[:10]:  # Top 10
        print(f"  {tf_info['TF']}: Rank {tf_info['rank']}")
    
    # Novel TF discovery
    top_20_tfs = consensus_df.head(20)['TF'].tolist()
    known_tf_names = [tf.lower() for tf in known_immune_tfs]
    
    novel_tfs = []
    for tf in top_20_tfs:
        is_known = any(known.lower() in tf.lower() for known in known_tf_names)
        if not is_known:
            novel_tfs.append(tf)
    
    validation_results['novel_tf_candidates'] = novel_tfs
    
    print(f"\nNovel TF candidates (top 20, not in known immune TF list):")
    for i, tf in enumerate(novel_tfs[:10], 1):
        rank = consensus_df[consensus_df['TF'] == tf].index[0] + 1
        print(f"  {i}. {tf} (Rank {rank})")
    
    return validation_results

# Placeholder for validation:
# validation_results = validate_tf_rankings(consensus_tf_ranking)

---

## Section 5: Results Summary and Therapeutic Target Prioritization

### Final Integration and Conclusions

This section will synthesize results from all three analytical approaches to provide:

#### Key Deliverables
1. **Consensus TF Ranking**: Top 50 transcription factors driving M(KP) conversion
2. **Novel Discovery Summary**: Previously unknown M(KP) regulators identified
3. **Minimal TF Combinations**: Optimized gene sets for therapeutic targeting
4. **Therapeutic Prioritization**: Druggable targets ranked by importance and feasibility
5. **Updated Mechanistic Model**: Enhanced understanding of M(KP) regulatory networks

#### Visualization Plan
- **Integrated TF ranking heatmap**: Scores across all methods
- **Network diagram**: Top TFs and their predicted target relationships
- **Trajectory TF dynamics**: Activity changes along pseudotime
- **ML feature importance**: Comparative model performance
- **Validation summary**: Known vs novel TF discovery

#### Impact Assessment
This analysis transforms the project from **validation of known biology** to **discovery of novel therapeutic targets**, addressing the key requirement for de novo transcription factor identification and ranking for cell state conversion.

### Next Steps for Experimental Validation
1. **Functional screening**: Test top-ranked TFs for M(KP) induction capability
2. **Combinatorial perturbation**: Validate minimal TF combinations
3. **Drug target assessment**: Evaluate therapeutic intervention feasibility
4. **Clinical translation**: Test TF signatures in human infection samples

---

## Implementation Notes

### Required Dependencies
```bash
# Core analysis
pip install scanpy pandas numpy matplotlib seaborn

# SCENIC (may require conda)
conda install -c bioconda pyscenic
# or: pip install pyscenic

# Machine learning
pip install scikit-learn

# Optional: advanced TF analysis
pip install cellrank scvelo
```

### Data Requirements
- **Input**: Processed AnnData from Notebook 6 with trajectory analysis
- **TF databases**: Mouse transcription factor gene lists
- **Motif databases**: JASPAR, ENCODE binding site predictions
- **Output**: Integrated TF rankings and validation results

### Computational Resources
- **SCENIC**: Computationally intensive, consider parallel processing
- **ML training**: Moderate requirements, benefits from multiple cores
- **Memory**: ~8-16GB recommended for full analysis
- **Runtime**: 2-4 hours for complete workflow

---

*This notebook establishes a comprehensive framework for transcription factor discovery and ranking in M(KP) macrophage polarization, extending the project's scope from validation to novel therapeutic target identification.*

In [ ]:
# Final Summary and Visualization Functions

def create_tf_ranking_summary(consensus_df, validation_results, output_dir='figures/'):
    """
    Create comprehensive summary visualizations of TF ranking results
    
    Parameters:
    - consensus_df: integrated TF ranking DataFrame
    - validation_results: validation against known biology
    - output_dir: directory for saving figures
    """
    print("Creating TF ranking summary visualizations...")
    
    # 1. Top 20 TF ranking heatmap
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Heatmap of method scores
    top_20 = consensus_df.head(20)
    score_matrix = top_20[['scenic_score', 'trajectory_score', 'ml_score']].values
    
    sns.heatmap(score_matrix, 
                xticklabels=['SCENIC', 'Trajectory', 'ML'],
                yticklabels=top_20['TF'],
                annot=True, fmt='.2f', cmap='viridis',
                ax=axes[0,0])
    axes[0,0].set_title('Top 20 TFs: Method Scores')
    
    # Integrated score ranking
    axes[0,1].barh(range(20), top_20['integrated_score'][::-1])
    axes[0,1].set_yticks(range(20))
    axes[0,1].set_yticklabels(top_20['TF'][::-1])
    axes[0,1].set_xlabel('Integrated Score')
    axes[0,1].set_title('Top 20 TFs: Integrated Ranking')
    
    # Methods detection count
    method_counts = consensus_df['methods_detected'].value_counts().sort_index()
    axes[1,0].bar(method_counts.index, method_counts.values)
    axes[1,0].set_xlabel('Number of Methods Detecting TF')
    axes[1,0].set_ylabel('Number of TFs')
    axes[1,0].set_title('TF Detection Consistency')
    
    # Novel vs known TFs in top rankings
    if 'novel_tf_candidates' in validation_results:
        novel_count = len(validation_results['novel_tf_candidates'])
        known_count = 20 - novel_count
        
        axes[1,1].pie([known_count, novel_count], 
                     labels=['Known Immune TFs', 'Novel Candidates'],
                     autopct='%1.1f%%')
        axes[1,1].set_title('Top 20 TFs: Known vs Novel')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}tf_ranking_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Create summary table
    summary_table = top_20[['TF', 'integrated_score', 'methods_detected', 
                           'scenic_score', 'trajectory_score', 'ml_score']].copy()
    
    # Add validation annotations
    summary_table['category'] = 'Novel'
    
    if validation_results.get('STAT6_rank'):
        stat6_tfs = [tf for tf in summary_table['TF'] if 'Stat6' in tf]
        summary_table.loc[summary_table['TF'].isin(stat6_tfs), 'category'] = 'STAT6 pathway'
    
    known_immune_tfs = [tf_info['TF'] for tf_info in validation_results.get('known_immune_tfs', [])]
    summary_table.loc[summary_table['TF'].isin(known_immune_tfs), 'category'] = 'Known immune'
    
    print("\n=== TOP 20 TRANSCRIPTION FACTORS FOR M(KP) CONVERSION ===")
    print(summary_table.to_string(index=False, float_format='%.3f'))
    
    return summary_table

# Placeholder for final summary:
# summary_table = create_tf_ranking_summary(consensus_tf_ranking, validation_results)

---

## Notebook Completion Checklist

### Data Loading and Preparation
- [ ] Load processed AnnData from Notebook 6
- [ ] Filter for Interstitial Macrophages (IM)
- [ ] Prepare expression matrices for each analysis method
- [ ] Validate data quality and cell annotations

### Section 1: SCENIC Analysis
- [ ] Install and configure SCENIC dependencies
- [ ] Run GENIE3 network inference
- [ ] Perform motif-based regulon pruning
- [ ] Calculate AUCell regulon activity scores
- [ ] Rank TFs by M(KP)-specific activity

### Section 2: Trajectory TF Analysis
- [ ] Calculate TF activity scores along trajectories
- [ ] Analyze pseudotime TF dynamics
- [ ] Identify branch point TF changes
- [ ] Rank TFs by transition influence

### Section 3: Machine Learning Analysis
- [ ] Prepare M(KP) vs other IM classification dataset
- [ ] Train multiple ML models
- [ ] Extract feature importance scores
- [ ] Identify minimal TF combinations

### Section 4: Integration and Validation
- [ ] Integrate rankings from all methods
- [ ] Validate against known M(KP) biology
- [ ] Identify novel TF candidates
- [ ] Assess therapeutic targeting potential

### Section 5: Results and Visualization
- [ ] Create comprehensive summary visualizations
- [ ] Generate final TF ranking table
- [ ] Document novel discoveries
- [ ] Prepare results for validation experiments

---

*Upon completion, this notebook will successfully demonstrate de novo transcription factor discovery and ranking for M(KP) cell state conversion, meeting the specified computational requirements and extending the project's impact beyond literature validation to novel therapeutic target identification.*